In [1]:
# Parameters
frequency = "1d"
window_pred = 7


In [2]:
import numpy as np
import pandas as pd
from pylab import plt, mpl
from sklearn.metrics import accuracy_score
import os
import papermill
import talib as ta
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPClassifier
import tensorflow as tf
from keras.layers import Dense
from keras.models import Sequential
from sklearn.inspection import permutation_importance

# Frecuencia obtenida desde el main
try:
    print(f"Frecuencia recibida desde papermill: {frequency}")
except NameError:
    print(f"No se recibió 'frequency'.")


# Cargar los datos para esta frecuencia de un archivo creado por el main
file_name = f"processed_data_{frequency}_charac.csv"
data = pd.read_csv(file_name, index_col='timestamp')
data


Frecuencia recibida desde papermill: 1d


,BTCUSDT_1d,ETHUSDT_1d,XRPUSDT_1d,BNBUSDT_1d,SOLUSDT_1d,ADAUSDT_1d,TRXUSDT_1d,LINKUSDT_1d,AVAXUSDT_1d
timestamp,,,,,,,,,
2020-09-22,10529.61,344.21,0.23302,24.0468,2.9082,0.08146,0.02499,8.7401,5.3193
2020-09-23,10241.46,320.72,0.22164,22.8331,2.8548,0.07663,0.02486,7.6364,3.5350
2020-09-24,10736.32,348.97,0.23276,24.5745,3.1433,0.08254,0.02625,9.8700,4.6411
2020-09-25,10686.67,351.92,0.24154,24.6924,3.1937,0.09693,0.02714,10.7279,4.7134
2020-09-26,10728.60,353.92,0.24153,26.1998,3.1287,0.09547,0.02718,10.3169,4.5200
...,...,...,...,...,...,...,...,...,...
2024-12-28,95300.00,3404.00,2.18430,722.1300,195.5000,0.88950,0.25840,21.9900,37.7400
2024-12-29,93738.20,3356.48,2.09420,694.7100,189.9400,0.85900,0.25780,20.9600,35.8400
2024-12-30,92792.05,3361.84,2.05870,705.3600,191.3800,0.86150,0.25340,20.5800,35.9700


Función para guardar los datos. Hace un archivo por cada frecuencia. Guarda en cada línea el modelo que se ha empleado, el activo, accuracy e in/out-sample.

In [3]:
def save_results(model, ric, acc, sample, frequency=frequency):
    # Verificar si el archivo ya existe
    file_name = f'accuracy_results_{frequency}_charac.csv'

    # Si el archivo existe, leer los datos previos, si no, crear un nuevo DataFrame vacío
    if os.path.exists(file_name):
        df_results = pd.read_csv(file_name)
    else:
        df_results = pd.DataFrame(columns=['Model', 'Asset', 'Accuracy', 'IN/OUT Sample'])

    # Agregar la nueva fila con los resultados
    new_row = pd.DataFrame([[model, ric, acc, sample]], columns=['Model', 'Asset', 'Accuracy', 'IN/OUT Sample'])
    df_results = pd.concat([df_results, new_row], ignore_index=True)

    # Guardar los resultados acumulados
    df_results.to_csv(file_name, index=False) 

Creamos las características que usaremos para hacer el aprendizaje ahora y las retardamos.

In [4]:
def charact_lags(data, ric, lags, window_pred, window=30):
    cols = []
    df = pd.DataFrame(data[ric])
    df.dropna(inplace=True)
    df['r'] = np.log(df / df.shift()) #retornos
    df['sma'] = df[ric].rolling(window).mean()  #media movil de la ventana
    df['min'] = df[ric].rolling(window).min() #mínimo de la ventana
    df['max'] = df[ric].rolling(window).max() #máximo de la ventana
    df['mom'] = df[ric].pct_change(window) #momentum de la ventana pct_change(12)
    df['vol'] = df['r'].rolling(window).std() #volatilidad de la ventana
    df['rsi'] = ta.RSI(df[ric], timeperiod=window) #rsi de la ventana
    df['atr'] = ta.ATR(df[ric], df[ric], df[ric], timeperiod=window) #atr de la ventana
    df.dropna(inplace=True)
    df = df.iloc[:-window_pred]
    df['d'] = np.where(df[ric].shift(-window_pred) > df[ric], 1, 0) # columna binaria, 0 si los precios bajarán, 1 si subirán
    df['ten'] = np.where(df[ric].shift(window_pred) > df[ric], 1, 0)
    print(df['d'].value_counts(normalize=True)) #comprueba si los datos están desbalanceados 
    features = [ric, 'r', 'ten', 'sma', 'min', 'max', 'mom', 'vol', 'rsi', 'atr']
    for f in features:
        for lag in range(1, lags + 1):
            col = f'{f}_lag_{lag}'
            df[col] = df[f].shift(lag)
            cols.append(col)
    df.dropna(inplace=True)
    return df, cols

lags = 5

dfs = {}
for ric in data:
    df, cols = charact_lags(data, ric, lags, window_pred)
    dfs[ric] = df.dropna(), cols

d
1    0.535387
0    0.464613
Name: proportion, dtype: float64


d
1    0.527523
0    0.472477
Name: proportion, dtype: float64
d
0    0.52228
1    0.47772
Name: proportion, dtype: float64
d
1    0.527523
0    0.472477
Name: proportion, dtype: float64


d
1    0.513761
0    0.486239
Name: proportion, dtype: float64


d
0    0.523591
1    0.476409
Name: proportion, dtype: float64


d
1    0.572739
0    0.427261
Name: proportion, dtype: float64
d
1    0.50983
0    0.49017
Name: proportion, dtype: float64
d
0    0.515072
1    0.484928
Name: proportion, dtype: float64


Hacemos una función que entrene el modelo, lo valide utilizando walk-forward y calcule el accuracy.

In [5]:
def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else: period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }

        results = []
        for ric in data:
            df, cols = dfs[ric]
            df = df[cols + ['d']]
            df['timestamp'] = pd.to_datetime(df.index)
            max_time = df['timestamp'].max()
            cutoff = max_time - final_test_period  # Reservamos el último año
            df_trainval = df[df['timestamp'] < cutoff]

            # Walk-forward en datos previos al último año
            min_time = df_trainval['timestamp'].min()
            split_dates = []
            current_time = min_time + period
            while current_time < cutoff:
                split_dates.append(current_time)
                current_time += period
            split_dates = split_dates[-5:]

            for split_date in split_dates:
                train = df_trainval[df_trainval['timestamp'] < (split_date - pd.Timedelta(days=window_pred))]
                test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]
                fin_train = split_date - pd.Timedelta(days=window_pred)
                #print('split_date, comienzo split', split_date)
                fin_test= split_date + period
                #print('fin_train', fin_train - pd.Timedelta(days=window_pred))
                #print('fin_test', fin_test )
                #print('\n')

                if len(test) == 0:
                    continue

                X_train, y_train = train.drop(columns=['d', 'timestamp']), train['d']
                X_test, y_test = test.drop(columns=['d', 'timestamp']), test['d']

                mean, std = X_train.mean(), X_train.std()
                std.replace(0, 1, inplace=True)
                X_train = (X_train - mean) / std
                X_test = (X_test - mean) / std

                model = model_class(**trial_params)
                model.fit(X_train, y_train)

                # importancia de las características
                '''result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
                sorted_idx = result.importances_mean.argsort()[::-1]  # orden descendente
                print("Feature importances (top 10):")
                for i in sorted_idx[:10]:
                    print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")'''

                pred = np.where(model.predict(X_test) > 0.5, 1, 0)
                acc = accuracy_score(y_test, pred)
                results.append(acc)

            avg_acc = np.mean(results)
            print(f'VALIDATION | {ric:7s} | acc={avg_acc:.4f}')
            save_results(model_class.__name__, ric, avg_acc, "OUT-SAMPLE")
        return avg_acc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # Entrenamiento final con mejor hiperparámetros, test en último año
    for ric in data:
        df, cols = dfs[ric]
        df = df[cols + ['d']]
        df['timestamp'] = pd.to_datetime(df.index)
        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period

        train = df[df['timestamp'] < (cutoff - pd.Timedelta(days=window_pred))]
        test = df[df['timestamp'] >= cutoff]
        

        if len(test) == 0:
            continue

        X_train, y_train = train.drop(columns=['d', 'timestamp']), train['d']
        X_test, y_test = test.drop(columns=['d', 'timestamp']), test['d']

        mean, std = X_train.mean(), X_train.std()
        std.replace(0, 1, inplace=True)
        X_train = (X_train - mean) / std
        X_test = (X_test - mean) / std

        model = model_class(
            hidden_layer_sizes=(best_params["hidden_units"],),
            alpha=best_params["alpha"],
            learning_rate_init=best_params["learning_rate"],
            max_iter=model_params.get("max_iter", 1000),
            early_stopping=model_params.get("early_stopping", True),
            validation_fraction=model_params.get("validation_fraction", 0.15),
            shuffle=model_params.get("shuffle", False),
            random_state=model_params.get("random_state", 100),
        )
        model.fit(X_train, y_train)
        # importancia de las características
        result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
        sorted_idx = result.importances_mean.argsort()[::-1]  # orden descendente
        print("Feature importances (top 10):")
        for i in sorted_idx[:10]:
            print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")
            
        pred = np.where(model.predict(X_test) > 0.5, 1, 0)
        acc = accuracy_score(y_test, pred)
        print(f'FINAL TEST | {ric:7s} | acc={acc:.4f}')
        save_results(model_class.__name__, ric, acc, "FINAL-TEST")

    return best_params


MODELO GLOBAL

In [6]:
'''def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else:
        period = pd.Timedelta(days=90)

    final_test_period = pd.Timedelta(days=365)

    def objective(trial):
        try:
            trial_params = {
                "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
                "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
                "learning_rate_init": trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True),
                "max_iter": model_params.get("max_iter", 1000),
                "early_stopping": model_params.get("early_stopping", True),
                "validation_fraction": model_params.get("validation_fraction", 0.15),
                "shuffle": model_params.get("shuffle", False),
                "random_state": model_params.get("random_state", 100),
            }

            results = []
            for split_date in get_split_dates(period, final_test_period):
                global_train, global_test = [], []

                for ric in data:
                    df, cols = dfs[ric]
                    df = df[cols + ['d']]
                    df['timestamp'] = pd.to_datetime(df.index)
                    cutoff = df['timestamp'].max() - final_test_period
                    df_trainval = df[df['timestamp'] < cutoff]

                    train = df_trainval[df_trainval['timestamp'] < split_date]
                    test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]

                    if len(test) == 0 or len(train) == 0:
                        continue

                    global_train.append(train)
                    global_test.append(test)

                if not global_train or not global_test:
                    print("[Trial Skipped] No se pudo generar train/test global.")
                    return None

                train_df = pd.concat(global_train)
                test_df = pd.concat(global_test)

                X_train, y_train = train_df.drop(columns=['d', 'timestamp']), train_df['d']
                X_test, y_test = test_df.drop(columns=['d', 'timestamp']), test_df['d']

                mean, std = X_train.mean(), X_train.std()
                std.replace(0, 1, inplace=True)
                X_train = (X_train - mean) / std
                X_test = (X_test - mean) / std

                X_train = X_train.fillna(X_train.mean())
                X_test = X_test.fillna(X_train.mean())

                model = model_class(**trial_params)
                model.fit(X_train, y_train)

                pred = np.where(model.predict(X_test) > 0.5, 1, 0)
                acc = accuracy_score(y_test, pred)
                results.append(acc)

            if not results:
                print("[Trial Skipped] No se generaron métricas.")
                return None

            avg_acc = np.mean(results)
            print(f'[GLOBAL MODEL] acc={avg_acc:.4f}')
            save_results(model_class.__name__, "GLOBAL", acc, "HIPERPARAM-TRAIN")
            return avg_acc

        except Exception as e:
            print(f"[Trial Failed] {e}")
            return None

    def get_split_dates(period, final_test_period):
        all_timestamps = [pd.to_datetime(dfs[ric][0].index) for ric in data]
        min_time = max(min(ts) for ts in all_timestamps)
        max_time = min(max(ts) for ts in all_timestamps)
        cutoff = max_time - final_test_period

        split_dates = []
        current_time = min_time + period
        while current_time < cutoff:
            split_dates.append(current_time)
            current_time += period
        return split_dates[-5:]

    # Optuna
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # ENTRENAMIENTO FINAL
    global_train, global_test = [], []
    for ric in data:
        df, cols = dfs[ric]
        df = df[cols + ['d']]
        df['timestamp'] = pd.to_datetime(df.index)

        cutoff = df['timestamp'].max() - final_test_period
        train = df[df['timestamp'] < cutoff]
        test = df[df['timestamp'] >= cutoff]

        if len(test) == 0:
            continue

        global_train.append(train)
        global_test.append(test)

    train_df = pd.concat(global_train)
    test_df = pd.concat(global_test)

    X_train, y_train = train_df.drop(columns=['d', 'timestamp']), train_df['d']
    X_test, y_test = test_df.drop(columns=['d', 'timestamp']), test_df['d']

    mean, std = X_train.mean(), X_train.std()
    std.replace(0, 1, inplace=True)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std
    X_train = X_train.fillna(X_train.mean())
    X_test = X_test.fillna(X_train.mean())

    model = model_class(
        hidden_layer_sizes=(best_params["hidden_units"],),
        alpha=best_params["alpha"],
        learning_rate_init=best_params["learning_rate"],
        max_iter=model_params.get("max_iter", 1000),
        early_stopping=model_params.get("early_stopping", True),
        validation_fraction=model_params.get("validation_fraction", 0.15),
        shuffle=model_params.get("shuffle", False),
        random_state=model_params.get("random_state", 100),
    )
    model.fit(X_train, y_train)
    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
    acc = accuracy_score(y_test, pred)

    print(f'[GLOBAL FINAL TEST] acc={acc:.4f}')
    save_results(model_class.__name__, "GLOBAL", acc, "FINAL-TEST")

    return best_params
'''

'def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5):\n    if freq == \'1h\':\n        period = pd.Timedelta(days=7)\n    elif freq == \'4h\':\n        period = pd.Timedelta(days=15)\n    else:\n        period = pd.Timedelta(days=90)\n\n    final_test_period = pd.Timedelta(days=365)\n\n    def objective(trial):\n        try:\n            trial_params = {\n                "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),\n                "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),\n                "learning_rate_init": trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True),\n                "max_iter": model_params.get("max_iter", 1000),\n                "early_stopping": model_params.get("early_stopping", True),\n                "validation_fraction": model_params.get("validation_fraction", 0.15),\n                "shuffle": model_params.get("shuffle", False),\n                "random_state": model_params.get("rand

Modelo MLP Classifier

In [7]:
# Ejecutar la optimización
model_params = {
    "max_iter": 1000,
    "early_stopping": True,
    "validation_fraction": 0.15,
    "shuffle": False,
    "random_state": 100
}

tuned_params = walk_forward_fit_test(MLPClassifier, frequency, model_params, n_trials=5)



[I 2025-05-04 18:32:29,621] A new study created in memory with name: no-name-e968ef79-0496-4b9b-a1c9-c6cb4e35cf2a


C:\Users\raque\AppData\Local\Temp\ipykernel_32996\2220521385.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
C:\Users\raque\AppData\Local\Temp\ipykernel_32996\2220521385.py:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
C:\Users\raque\AppData\Local\Temp\ipykernel_32996\2220521385.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexi

VALIDATION | BTCUSDT_1d | acc=0.5458


C:\Users\raque\AppData\Local\Temp\ipykernel_32996\3694455987.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, new_row], ignore_index=True)
C:\Users\raque\AppData\Local\Temp\ipykernel_32996\2220521385.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['timestamp'] = pd.to_datetime(df.index)


VALIDATION | BTCUSDT_1d | acc=0.4876


...

Feature importances (top 10):
TRXUSDT_1d_lag_1               - Importance: 0.0673
rsi_lag_4                      - Importance: 0.0645
max_lag_1                      - Importance: 0.0500
min_lag_2                      - Importance: 0.0474
max_lag_4                      - Importance: 0.0446
min_lag_3                      - Importance: 0.0438
max_lag_5                      - Importance: 0.0438
atr_lag_3                      - Importance: 0.0388
sma_lag_2                      - Importance: 0.0370
rsi_lag_5                      - Importance: 0.0328
FINAL TEST | TRXUSDT_1d | acc=0.6120


C:\Users\raque\AppData\Local\Temp\ipykernel_32996\2220521385.py:89: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['timestamp'] = pd.to_datetime(df.index)


Feature importances (top 10):
ten_lag_5                      - Importance: 0.0183
LINKUSDT_1d_lag_2              - Importance: 0.0145
ten_lag_4                      - Importance: 0.0133
min_lag_5                      - Importance: 0.0128
LINKUSDT_1d_lag_1              - Importance: 0.0128
ten_lag_1                      - Importance: 0.0127
min_lag_3                      - Importance: 0.0118
rsi_lag_3                      - Importance: 0.0109
sma_lag_1                      - Importance: 0.0107
atr_lag_3                      - Importance: 0.0104
FINAL TEST | LINKUSDT_1d | acc=0.4945


C:\Users\raque\AppData\Local\Temp\ipykernel_32996\2220521385.py:89: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['timestamp'] = pd.to_datetime(df.index)


Feature importances (top 10):
mom_lag_4                      - Importance: 0.0197
AVAXUSDT_1d_lag_1              - Importance: 0.0170
rsi_lag_4                      - Importance: 0.0162
ten_lag_1                      - Importance: 0.0149
max_lag_5                      - Importance: 0.0117
atr_lag_3                      - Importance: 0.0107
vol_lag_2                      - Importance: 0.0100
rsi_lag_3                      - Importance: 0.0087
r_lag_2                        - Importance: 0.0087
atr_lag_1                      - Importance: 0.0084
FINAL TEST | AVAXUSDT_1d | acc=0.4590


Modelo Bagging Classifier

In [8]:
'''
def walk_forward_fit_test(freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else: period = pd.Timedelta(days=90)
    
    final_test_period = pd.Timedelta(days=365)

    def objective(trial):
        base_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }

        results = []
        for ric in data:
            df, cols = dfs[ric]
            df = df[cols + ['d']]
            df['timestamp'] = pd.to_datetime(df.index)
            max_time = df['timestamp'].max()
            cutoff = max_time - final_test_period
            df_trainval = df[df['timestamp'] < cutoff]

            min_time = df_trainval['timestamp'].min()
            split_dates = []
            current_time = min_time + period
            while current_time < cutoff:
                split_dates.append(current_time)
                current_time += period
            split_dates = split_dates[-5:]

            for split_date in split_dates:
                train = df_trainval[df_trainval['timestamp'] < split_date]
                test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]

                if len(test) == 0:
                    continue

                X_train, y_train = train.drop(columns=['d', 'timestamp']), train['d']
                X_test, y_test = test.drop(columns=['d', 'timestamp']), test['d']

                mean, std = X_train.mean(), X_train.std()
                std.replace(0, 1, inplace=True)
                X_train = (X_train - mean) / std
                X_test = (X_test - mean) / std

                base_model = MLPClassifier(**base_params)
                model = BaggingClassifier(base_estimator=base_model, n_estimators=5, random_state=42)
                model.fit(X_train, y_train)

                pred = np.where(model.predict(X_test) > 0.5, 1, 0)
                acc = accuracy_score(y_test, pred)
                results.append(acc)

        avg_acc = np.mean(results)
        print(f'OUT-OF-SAMPLE | {ric:7s} | acc={avg_acc:.4f}')
        save_results("BaggingMLP", ric, avg_acc, "OUT-SAMPLE")
        return avg_acc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    for ric in data:
        df, cols = dfs[ric]
        df = df[cols + ['d']]
        df['timestamp'] = pd.to_datetime(df.index)

        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period

        train = df[df['timestamp'] < cutoff]
        test = df[df['timestamp'] >= cutoff]

        if len(test) == 0:
            continue

        X_train, y_train = train.drop(columns=['d', 'timestamp']), train['d']
        X_test, y_test = test.drop(columns=['d', 'timestamp']), test['d']

        mean, std = X_train.mean(), X_train.std()
        std.replace(0, 1, inplace=True)
        X_train = (X_train - mean) / std
        X_test = (X_test - mean) / std

        base_model = MLPClassifier(
            hidden_layer_sizes=(best_params["hidden_units"],),
            alpha=best_params["alpha"],
            learning_rate_init=best_params["learning_rate"],
            max_iter=model_params.get("max_iter", 1000),
            early_stopping=model_params.get("early_stopping", True),
            validation_fraction=model_params.get("validation_fraction", 0.15),
            shuffle=model_params.get("shuffle", False),
            random_state=model_params.get("random_state", 100),
        )

        model = BaggingClassifier(base_estimator=base_model, n_estimators=5, random_state=42)
        model.fit(X_train, y_train)

        pred = np.where(model.predict(X_test) > 0.5, 1, 0)
        acc = accuracy_score(y_test, pred)
        print(f'FINAL TEST | {ric:7s} | acc={acc:.4f}')
        save_results("BaggingMLP", ric, acc, "FINAL-TEST")

    return best_params'''


'\ndef walk_forward_fit_test(freq, model_params={}, n_trials=5):\n    if freq == \'1h\':\n        period = pd.Timedelta(days=7)\n    elif freq == \'4h\':\n        period = pd.Timedelta(days=15)\n    else: period = pd.Timedelta(days=90)\n    \n    final_test_period = pd.Timedelta(days=365)\n\n    def objective(trial):\n        base_params = {\n            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),\n            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),\n            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),\n            "max_iter": model_params.get("max_iter", 1000),\n            "early_stopping": model_params.get("early_stopping", True),\n            "validation_fraction": model_params.get("validation_fraction", 0.15),\n            "shuffle": model_params.get("shuffle", False),\n            "random_state": model_params.get("random_state", 100),\n        }\n\n        results = []\n        for ric in data:

In [9]:
'''from sklearn.ensemble import BaggingClassifier
from sklearn.neural_network import MLPClassifier   

# Definir base_estimator
base_estimator = MLPClassifier(tuned_params) 

# Ejecutar la optimización
tuned_params_b = walk_forward_fit_test(BaggingClassifier, frequency, {"base_estimator": base_estimator},  
                            n_trials=10)'''

'from sklearn.ensemble import BaggingClassifier\nfrom sklearn.neural_network import MLPClassifier   \n\n# Definir base_estimator\nbase_estimator = MLPClassifier(tuned_params) \n\n# Ejecutar la optimización\ntuned_params_b = walk_forward_fit_test(BaggingClassifier, frequency, {"base_estimator": base_estimator},  \n                            n_trials=10)'